In [1]:
import pandas as pd

file_path = 'metadata.csv' 

# low_memory=False --> złożony plik
df = pd.read_csv(file_path, low_memory=False)


### podstawowa analiza
print("Kształt danych (wiersze, kolumny):")
print(df.shape)
print("-" * 30)

print("Pierwsze 5 wierszy:")
print(df.head())
print("-" * 30)

print("Info o typach danych:")
df.info()
print("-" * 30)


print("Kolumny:")
print(df.columns)
print("-" * 30)

print("Liczba brakujących wartości w kolumnach:")
print(df.isnull().sum())

Kształt danych (wiersze, kolumny):
(1056660, 19)
------------------------------
Pierwsze 5 wierszy:
   cord_uid                                       sha source_x  \
0  ug7v899j  d1aafb70c066a2068b02786f8929fd9c900897fb      PMC   
1  02tnwd4m  6b0567729c2143a66d737eb0a2f63f2dce2e5a7d      PMC   
2  ejv2xln0  06ced00a5fc04215949aa72528f2eeaae1d58927      PMC   
3  2b73a28n  348055649b6b8cf2b9a376498df9bf41f7123605      PMC   
4  9785vg6d  5f48792a5fa08bed9f56016f4981ae2ca6031b32      PMC   

                                               title                    doi  \
0  Clinical features of culture-proven Mycoplasma...  10.1186/1471-2334-1-6   
1  Nitric oxide: a pro-inflammatory mediator in l...           10.1186/rr14   
2    Surfactant protein-D and pulmonary host defense           10.1186/rr19   
3               Role of endothelin-1 in lung disease           10.1186/rr44   
4  Gene expression in epithelial cells in respons...           10.1186/rr61   

      pmcid pubmed_id licens

In [2]:
print(f"Rozmiar danych przed czyszczeniem: {df.shape}")

### tylko potrzebne kolumny: title i abstract
### .copy()  -->  nowa niezależna ramka danych
df_clean = df[['title', 'abstract']].copy()

### usunięcie wierszy, w których brak jednoczesnie tytułu i abstraktu
### wiersz zostanie usunięty tylko jeśli wszystkie wartości w podanych kolumnach subset są puste (NaN)
df_clean.dropna(subset=['title', 'abstract'], how='all', inplace=True)

print(f"Rozmiar po usunięciu wierszy bez tytułu i abstraktu: {df_clean.shape}")

### usunięcie duplikatów na podstawie kolumny title
df_clean.drop_duplicates(subset=['title'], inplace=True)

print(f"Rozmiar po usunięciu duplikatów: {df_clean.shape}")


print("\nLiczba brakujących wartości po czyszczeniu:")
print(df_clean.isnull().sum())


print("\nPróbka danych po czyszczeniu:")
print(df_clean.head())

Rozmiar danych przed czyszczeniem: (1056660, 19)
Rozmiar po usunięciu wierszy bez tytułu i abstraktu: (1056268, 2)
Rozmiar po usunięciu duplikatów: (850367, 2)

Liczba brakujących wartości po czyszczeniu:
title            1
abstract    176878
dtype: int64

Próbka danych po czyszczeniu:
                                               title  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2    Surfactant protein-D and pulmonary host defense   
3               Role of endothelin-1 in lung disease   
4  Gene expression in epithelial cells in respons...   

                                            abstract  
0  OBJECTIVE: This retrospective chart review des...  
1  Inflammatory diseases of the respiratory tract...  
2  Surfactant protein-D (SP-D) participates in th...  
3  Endothelin-1 (ET-1) is a 21 amino acid peptide...  
4  Respiratory syncytial virus (RSV) and pneumoni...  


In [3]:
import re

### zastąpienie wartości NaN pustym tekstem ''
df_clean.fillna('', inplace=True)

### nowa połączona kolumna 'text' ### spacja ' ' pomiędzy aby tytuł nie skleił się z abstraktem
df_clean['text'] = df_clean['title'] + ' ' + df_clean['abstract']

### funkcja do czyszczenia tekstu
def preprocess_text(text):
    text = text.lower()  ## zamiana na małe litery
    text = re.sub(r'[^\w\s]', '', text)  ## usunięcie interpunkcji
    text = re.sub(r'\d+', '', text)  ## usunięcie cyfr
    text = re.sub(r'\s+', ' ', text).strip() ## usunięcie nadmiarowych białych znaków
    return text


print("Czyszczenie tekstu...")
df_clean['text_clean'] = df_clean['text'].apply(preprocess_text)
print("Czyszczenie zakończone.")

### 5 wierszy z kolumną oryginalną i wyczyszczoną dla porównania
print("\nPorównanie tekstu oryginalnego i wyczyszczonego:")
print(df_clean[['text', 'text_clean']].head())

Czyszczenie tekstu...
Czyszczenie zakończone.

Porównanie tekstu oryginalnego i wyczyszczonego:
                                                text  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2  Surfactant protein-D and pulmonary host defens...   
3  Role of endothelin-1 in lung disease Endotheli...   
4  Gene expression in epithelial cells in respons...   

                                          text_clean  
0  clinical features of cultureproven mycoplasma ...  
1  nitric oxide a proinflammatory mediator in lun...  
2  surfactant proteind and pulmonary host defense...  
3  role of endothelin in lung disease endothelin ...  
4  gene expression in epithelial cells in respons...  


In [4]:
import spacy
from tqdm import tqdm
import time

### preprocessing z Lematyzacją
nlp = spacy.load('en_core_web_lg', disable=['parser', 'ner']) ## wyłącza komponenty (parser, ner) aby przyspieszyć proces
print("Model załadowany.")

## funkcja do lematyzacji i usuwania stop words
def lemmatize_text(text):
    doc = nlp(text)
    
    # lista lemmatów bez stop words i znaków interpunkcyjnych
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    
    # łączy lemmaty z powrotem w jeden tekst
    return " ".join(lemmas)

## mniejsza próba
print("\nPrzetwarzanie próby 10000 dokumentów...")
df_sample = df_clean.sample(10000, random_state=42)

start_time = time.time()

df_sample_lemmatized = df_sample.copy()
df_sample_lemmatized['text_lemmatized'] = df_sample_lemmatized['text_clean'].apply(lemmatize_text)

end_time = time.time()
print(f"Przetwarzanie próbki zakończone w {(end_time - start_time):.2f} sekund.")


## wyniki
print("\nPorównanie tekstu wyczyszczonego i zlemmatyzowanego:")
print(df_sample_lemmatized[['text_clean', 'text_lemmatized']].head())

Model załadowany.

Przetwarzanie próby 10000 dokumentów...
Przetwarzanie próbki zakończone w 105.69 sekund.

Porównanie tekstu wyczyszczonego i zlemmatyzowanego:
                                               text_clean  \
666494  potent noncovalent inhibitors of the main prot...   
304385  social study resources and social wellbeing be...   
143551  younger adults caught in covid crosshairs as d...   
796106  individuals conceptions of covid pandemic thro...   
283352  profiles of antibody responses against severe ...   

                                          text_lemmatized  
666494  potent noncovalent inhibitor main protease sar...  
304385  social study resource social wellbeing intelli...  
143551  young adult catch covid crosshair demographic ...  
796106  individual conception covid pandemic metaphor ...  
283352  profile antibody response severe acute respira...  


In [10]:
### przetwarzanie (lematyzacja) dla pełnego zbioru (wersja z nlp.pipe)

print("\nRozpoczynam przetwarzanie pełnego zbioru...")
start_time = time.time()


lemmatized_texts = []
### batch_size=500 spaCy przetwarza dokumenty po 500
for doc in tqdm(nlp.pipe(df_clean['text_clean'], batch_size=500), total=len(df_clean)):
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    lemmatized_texts.append(" ".join(lemmas))


df_clean['text_lemmatized'] = lemmatized_texts


end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nPrzetwarzanie pełnego zbioru zakończone w {total_time_minutes:.2f} minut.")


## wyniki
print("\nPorównanie tekstu wyczyszczonego i zlemmatyzowanego:")
print(df_clean[['text_clean', 'text_lemmatized']].head())


Rozpoczynam przetwarzanie pełnego zbioru...


100%|██████████| 850367/850367 [2:18:51<00:00, 102.07it/s]  



Przetwarzanie pełnego zbioru zakończone w 138.86 minut.

Porównanie tekstu wyczyszczonego i zlemmatyzowanego:
                                          text_clean  \
0  clinical features of cultureproven mycoplasma ...   
1  nitric oxide a proinflammatory mediator in lun...   
2  surfactant proteind and pulmonary host defense...   
3  role of endothelin in lung disease endothelin ...   
4  gene expression in epithelial cells in respons...   

                                     text_lemmatized  
0  clinical feature cultureproven mycoplasma pneu...  
1  nitric oxide proinflammatory mediator lung dis...  
2  surfactant proteind pulmonary host defense sur...  
3  role endothelin lung disease endothelin et ami...  
4  gene expression epithelial cell response pneum...  


In [12]:
import joblib

joblib.dump(df_clean, 'df_clean_lemmatized.joblib')
print("DataFrame zapisany w pliku df_clean_lemmatized.joblib")

DataFrame zapisany w pliku df_clean_lemmatized.joblib
